# SmolVLM spline + MMR versus MLP + MMR at block 3

Set `PREDICTOR_CHECKPOINT` to your trained after2 `predictor_best.pt`, or set a Hugging Face repository and revision with that file. Select **GPU T4 ×2**, enable Internet, and use **Save & Run All**. A Hugging Face predictor can be run without attaching a Kaggle Input.

Both variants keep 32 of 384 visual tokens immediately after decoder block 3 (`layers[2]`). Both run on the same 4,204 stratified DriveLM test questions with the same six-camera features, backbone, batch, FP32 greedy decoding, token budget, MMR λ = 0.9, min–max relevance normalization, and cosine redundancy from block-3 outputs.

- **Spline + MMR** subtracts a separate smoothing spline per camera from the attention at block 3. The fitted difference is block 3 minus block 23. Smoothing is selected on two disjoint recording groups from the 800 validation questions, then fit on all 800.
- **MLP + MMR** uses the frozen after2 MAP predictor from the original branch, with the last prompt query and the visual hidden states after block 3. It uses the existing trained weights and interprets its log probabilities as relevance probabilities before MMR.

The report includes paired answers, selected token indices, quality and latency under the same timing procedure, and the spline plots. Only the source of MMR relevance changes during evaluation.

In [ ]:
GPU_COUNT = 2
CALIBRATION_SIZE = 800
CALIBRATION_BATCH = 4  # Per GPU.
GENERATION_BATCH = 4   # Per GPU, with automatic CUDA OOM backoff.
KEEP_COUNTS = (32,)
CAMERA_BIAS_LAYERS = (3, 23)  # One-based decoder block numbers.
MMR_LAMBDA = 0.9
EVALUATION_SUBSET_SIZE = 4204
SELECTION_EXAMPLES = 3  # Save full ranked selections for the first evaluation examples.
CACHE_INPUT_DIR = ''   # Leave blank to fetch images and build features automatically.
PREDICTOR_CHECKPOINT = ''  # Exact local path to the trained after2 predictor_best.pt.
PREDICTOR_HF_REPO = ''  # Alternative Hugging Face location for the existing predictor.
PREDICTOR_HF_REVISION = ''  # Required when using the Hugging Face option.
PREDICTOR_HF_FILENAME = 'predictor_best.pt'
PREDICTOR_HF_REPO_TYPE = 'model'

In [ ]:
# Embedded source snapshot. Run this cell before importing Transformers.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
from pathlib import Path
from base64 import b64decode
from io import BytesIO
import shutil
import subprocess
import sys
import tempfile
import zipfile

if shutil.which('java') is None:
    raise RuntimeError('Java is required by the existing COCO caption evaluation metrics.')
SOURCE = Path(tempfile.mkdtemp(prefix='log-metrics-posprune-'))
SOURCE_B64 = '...'
with zipfile.ZipFile(BytesIO(b64decode(SOURCE_B64))) as archive:
    archive.extractall(SOURCE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt'), 'pycocoevalcap==1.2'], check=True)
sys.path.insert(0, str(SOURCE))

In [ ]:
import torch
import transformers
import tokenizers
from huggingface_hub import hf_hub_download
from smolvlm_attention.config import MODELS, CACHE_DATASET, CACHE_FINGERPRINT, PREPARED_REPO, PREPARED_REVISION
from smolvlm_attention.posprune_ablation.cache_input import resolve_cache_input

if (transformers.__version__, tokenizers.__version__) != ('4.46.3', '0.20.4'):
    raise RuntimeError('Restart the kernel after dependency installation and Run All.')
visible_gpus = torch.cuda.device_count()
if visible_gpus == 0:
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
if GPU_COUNT < 1:
    raise ValueError('GPU_COUNT must be at least 1.')
worker_count = min(GPU_COUNT, visible_gpus)
print(f'Using {worker_count} of {visible_gpus} visible GPU(s).')


if PREDICTOR_CHECKPOINT and PREDICTOR_HF_REPO:
    raise ValueError('Choose a local predictor checkpoint or a Hugging Face repository.')
if PREDICTOR_CHECKPOINT:
    predictor_checkpoint = Path(PREDICTOR_CHECKPOINT).resolve(strict=True)
elif PREDICTOR_HF_REPO and PREDICTOR_HF_REVISION:
    predictor_checkpoint = Path(hf_hub_download(repo_id=PREDICTOR_HF_REPO, revision=PREDICTOR_HF_REVISION,
        filename=PREDICTOR_HF_FILENAME, repo_type=PREDICTOR_HF_REPO_TYPE))
else:
    raise ValueError('Set the path to your trained after2 predictor_best.pt, or its Hugging Face repository and revision.')

cache_dir = None
if CACHE_INPUT_DIR:
    cache_dir = resolve_cache_input('/kaggle/input', CACHE_INPUT_DIR, dataset=CACHE_DATASET,
                                   fingerprint=CACHE_FINGERPRINT)
    print(f'Using mounted feature cache: {cache_dir}')
work = Path('/kaggle/working/smolvlm-posprune-spline-vs-mlp-after3-32')
prepared_dir = work / 'prepared_data'
for name in ('manifest.json', 'train.jsonl', 'val.jsonl', 'test.jsonl'):
    hf_hub_download(repo_id=PREPARED_REPO, repo_type='dataset', revision=PREPARED_REVISION,
                    filename=name, local_dir=prepared_dir, token=False)
base = MODELS['base']
checkpoint = hf_hub_download(repo_id=base['repo'], revision=base['revision'],
                             filename='finetune_best.pt', token=False)

In [ ]:
import json
cohort = json.loads((SOURCE / 'camera_posprune_cohort.json').read_bytes())

from smolvlm_attention.posprune_ablation.parallel import run_parallel

run_dir = Path(tempfile.mkdtemp(prefix='run-', dir=work))
output = run_parallel(checkpoint, cache_dir, prepared_dir, run_dir / 'results',
                      cache_fingerprint=CACHE_FINGERPRINT, camera_order=base['camera_order'], gpu_count=worker_count,
                      calibration_size=CALIBRATION_SIZE, evaluation_size=None, evaluation_fraction=cohort['evaluation_fraction'],
                      calibration_batch=CALIBRATION_BATCH, generation_batch=GENERATION_BATCH,
                      keep_counts=KEEP_COUNTS, seed=cohort['seed'],
                      selection_examples=SELECTION_EXAMPLES, camera_bias_layers=CAMERA_BIAS_LAYERS,
                      mmr_weight=MMR_LAMBDA, attention_normalization='minmax',
                      evaluation_subset_size=EVALUATION_SUBSET_SIZE, expected_evaluation_ids=cohort['evaluation_ids'],
                      predictor_checkpoint=predictor_checkpoint)
print((output / 'parallel_timing.json').read_text())
print(output)

In [ ]:
from IPython.display import FileLink, Image, display
from smolvlm_attention.posprune_ablation.report import report

for path in report(output):
    if path.suffix == '.png':
        display(Image(filename=str(path)))
archive = run_dir / 'posprune-results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(output.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
tables = [output / name for name in ('indexed_selections.csv', 'comparison.csv', 'paired_answers.csv',
                                      'question_type_distribution.csv')]
for path in (*tables, archive):
    display(FileLink(os.path.relpath(path)))

display(FileLink(os.path.relpath(output / 'camera_fit_metrics.csv')))